# 61. F1 0.5 목표 탐색 후보 감사
60번 실제예측으로전체8TP/6FP/F1.516과보수적앵커7TP/6FP/F1.467을재집계한다. RH정답·예측중복·기존탐지유지·고정2-of-3규칙·센서직전변화의원본이력일치·외부구간분리·과거평가미래제외를검사한다. 추가2개가과거평가대상인지도확인한다.
목표의보유데이터구간검증수치는달성했지만독립성능/대회공식점수/원인확정은아니다. 고정학습내부선택은개선하지못하고온도앵커54는참조수에민감하므로자동채택하지않는다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
d=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[(d['product']=='RG3')&((d.PassOrFail==0)|d.Reason.str.contains('가스',na=False))].copy()
rh=pd.read_csv(root/'data/audit/rg3_rh_labeled_with_timestamp.csv',parse_dates=['TimeStamp'])
rh=rh[(rh.PassOrFail==0)|rh.Reason.str.contains('가스',na=False)]
joined=d.merge(rh[['TimeStamp','PassOrFail','Reason']],on='TimeStamp',validate='one_to_one',suffixes=('_feature','_RH'))
assert len(joined)==583
assert (joined.PassOrFail_feature==joined.PassOrFail_RH).all() and (joined.Reason_feature==joined.Reason_RH).all()
p=pd.read_csv(tables/'rg3_pressure_seed_majority_predictions.csv',parse_dates=['TimeStamp'])
base=pd.read_csv(tables/'rg3_pdf_fixed_majority_predictions.csv',parse_dates=['TimeStamp'])
rows=[];gas_tables=[]
for (scheme,anchor_name),g in p.groupby(['scheme','anchor']):
 assert not g.TimeStamp.duplicated().any()
 gold=d.set_index('TimeStamp').loc[g.TimeStamp,'PassOrFail'].to_numpy(int)
 assert np.array_equal(gold,g.actual.to_numpy())
 assert ((g.anchor_prediction==0)|(g.prediction==1)).all()
 assert (g.prediction.astype(bool)==(g.anchor_prediction.astype(bool)|(g.positive_votes>=2))).all()
 assert g.positive_votes.between(0,3).all()
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(scheme=scheme,anchor=anchor_name,shots=len(g),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn),predicted_defect=int(tp+fp)))
 if scheme=='leave_one_block_out':
  x=g[g.actual.eq(1)].merge(base[base.scheme.eq(scheme)][['TimeStamp','prediction']],on='TimeStamp',validate='one_to_one',suffixes=('','_baseline57'))
  gas_tables.append(x)
v=pd.DataFrame(rows)
primary=v[(v.scheme=='leave_one_block_out')&(v.anchor=='exploratory54')].iloc[0]
assert (primary.TP,primary.FP,primary.FN,primary.TN)==(8,6,9,560)
assert primary.F1>=.5
conservative=v[(v.scheme=='leave_one_block_out')&(v.anchor=='majority57')].iloc[0]
assert (conservative.TP,conservative.FP)==(7,6)
# Rebuild original sensor history, inspecting current/previous only.
s10=''.join(next(c for c in json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))['cells'] if c['cell_type']=='code')['source'])
exec(s10.split('for feature in features:')[0])
delta_sensors=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Injection_Speed','Cushion_Position']
for name in delta_sensors:
 observed[name+'__audit_delta']=observed.groupby('history_segment')[name].diff()
 hist=observed[observed['product'].eq('RG3')][keys+[name+'__audit_delta']]
 check=d[keys+[name+'__delta1']].merge(hist,on=keys,validate='one_to_one')
 assert len(check)==583 and np.allclose(check[name+'__delta1'],check[name+'__audit_delta'],equal_nan=True)
fold=pd.read_csv(tables/'rg3_pressure_direction_fold_audit.csv')
blocklist=d.comparison_block.drop_duplicates().tolist()
for r in fold.itertuples():
 q=d[d.comparison_block.eq(r.block)]
 ref=d[~d.comparison_block.eq(r.block)] if r.scheme=='leave_one_block_out' else d[d.comparison_block.isin(blocklist[:blocklist.index(r.block)])]
 assert len(ref)==r.train and len(q)==r.test
 assert set(ref.TimeStamp).isdisjoint(q.TimeStamp)
 assert set(ref.comparison_block).isdisjoint(q.comparison_block)
 if r.scheme=='past_blocks_only':assert ref.TimeStamp.max()<q.TimeStamp.min()
gas=pd.concat(gas_tables,ignore_index=True)
past_times=set(p[p.scheme.eq('past_blocks_only')].TimeStamp)
gas['past_evaluation_eligible']=gas.TimeStamp.isin(past_times)
v.to_csv(tables/'rg3_f1target_verification.csv',index=False,encoding='utf-8-sig')
gas.to_csv(tables/'rg3_f1target_all17.csv',index=False,encoding='utf-8-sig')
p[p.anchor.eq('exploratory54')].to_csv(tables/'rg3_f1target_predictions.csv',index=False,encoding='utf-8-sig')
print(v.to_string(index=False))
print('New gas cases versus original majority57')
print(gas[(gas.anchor=='exploratory54')&(gas.prediction==1)&(gas.prediction_baseline57==0)][['TimeStamp','positive_votes','past_evaluation_eligible']].to_string(index=False))
print('Actual predictions, RH labels, current/previous sensor differences, all outer fold exclusions verified.')
print('CAVEAT: repeated/test-informed exploration; new cases are absent from forward-evaluation test; no causal proof or independent validation.')
